# Main Regression — The Spanish 15-min Reform Sequence

Graduates the thesis from descriptive patterns (nb03, nb05, nb06) to formal causal identification using Spain's sequence of 2024–2025 market-design reforms as natural experiments.

**Central thesis — the sequencing interpretation.** Spain reformed the imbalance-settlement side (ISP15, 2024-12-01) *before* reforming the intraday-trading side (MTU15-IDA, 2025-03-19). This creates a three-month window (Dec 2024 → Mar 2025) in which the *settlement* rules demanded 15-min precision but the *intraday market* still only offered 60-min instruments — a genuine granularity mismatch. Under this lens the two reforms operate on distinct economic margins:

- **ISP15** eliminates intra-hour imbalance netting. Before, a firm could run $+50$ MW in period 1 and $-50$ MW in period 4 of the same hour and the imbalances would net to zero at settlement; settlement cost was zero. After, each 15-min ISP is adjudicated separately. This *binds* firm strategy space — the ability to accept coarse DA / IDA commitments and let settlement smooth them away is gone.

- **MTU15-IDA** gives firms 15-min intraday trading tools to match the 15-min settlement. It does not impose a new constraint; it *relieves* the constraint ISP15 created. Firms can now fine-tune within-hour positions in a market that matches the settlement granularity.

**What each reform should do to the data under this story.**

- **ISP15 should produce a discrete behavioural shift.** It is the binding constraint. Under a DiD, it should be the identified step.
- **MTU15-IDA should be a softening, not a break.** It removes a friction rather than adding one. Under a DiD it may not produce a separately identified coefficient on the quantity outcome, but it should show up as a *bid-level* unwinding — conduct gaps and strategic premia should collapse once the granularity mismatch closes.
- **The reforms interact sequentially, not simultaneously.** The behaviourally important fact is not "MTU15-IDA exists" but "MTU15-IDA closes a gap that ISP15 opened."

**Identification strategy.** A unit-FE / date-FE difference-in-differences with unit-clustered SEs around each reform, plus a saturated specification that enters all four reform $\times$ Big-4 interactions simultaneously to isolate the marginal effect of each. The ISP15 coefficient is the main object of identification on the quantity outcome $\Delta Q$; MTU15-IDA reads as a relief mechanism (null on the quantity DiD, significant on the bid-level conduct-gap object from nb06 §2).

**Why this is well-identified in our setting.** Each reform happens at a *single* calendar date for all Spanish units simultaneously — no staggered adoption. The modern-DiD pathologies (Goodman–Bacon 2021 weighting problems, Callaway–Sant'Anna timing-cohort concerns, de Chaisemartin–d'Haultfœuille negative weights) arise when different units are treated at different dates; none of them apply here. Standard two-way fixed effects is the right estimator.

**What still can go wrong and how we test for it.**

1. **Pre-trends** — if Big-4 and the control group were already diverging before the reform date, the DiD attributes the pre-existing trend to the reform. **Tested in §3 via event-study coefficients; under the sequencing story the pre-trend may reflect anticipation of the pre-announced ISP15 reform starting mid-2024.**
2. **Confounding reforms** — the four reforms stack. **Tested in §5 via saturated reform-dummy specification where all four reform × Big-4 interactions enter jointly.**
3. **Control group comparability** — the natural Fringe pool includes 202 small RE hydro units with fundamentally different dispatch physics. **Tested in §8 by refining Fringe to dispatchable conventional only and by adding a within-Big-4 event-study with no Fringe.**
4. **Spurious time trend** — a placebo reform date in a stable regime should yield $\hat\beta \approx 0$. **Tested in §6.**
5. **Spillovers (SUTVA)** — Big-4 actions affect clearing prices and may cause Fringe to respond. **This biases $\hat\beta$ toward zero**, so our estimate is conservative.

**Scope.** Identification of ISP15 (primary) and characterisation of MTU15-IDA (as relief mechanism). The reduced-form bottom line: ISP15 has an identified discrete effect on Big-4 per-unit-day $\Delta Q$; MTU15-IDA does not add a separately identified step on the same outcome, consistent with it operating through bid-level adjustments (nb06 §2) rather than quantity-level repositioning. The theory in `theory/granularity_extension.tex` provides the two-channel framework: ISP15 activates the imbalance-gaming channel $(\pi - \alpha_r)$; MTU15-IDA deactivates the ramp-thinness channel $\Phi(\lambda, \beta, b_{21})$.

## § 0 — Setup and constants

Imports the econometrics stack (`linearmodels.PanelOLS` for two-way FE with clustered SEs; `statsmodels` for event-study plotting and auxiliary tests) and the project's shared reform dates from `mtu.notebook_utils`. The analysis window is 2023-12-01 onwards, consistent with nb03 and nb05.

In [ ]:
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from linearmodels.panel import PanelOLS

from mtu.notebook_utils import (
    PROJECT_ROOT,
    IDA_REFORM, ISP15_REFORM, INTRADAY_REFORM, DAY_AHEAD_REFORM,
)

# Ignore linearmodels' harmless RunTime warnings about absorbing the entity
# effect when we have fewer than 2 obs for some degenerate groups in subsamples.
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Data paths.
PDBC   = PROJECT_ROOT / 'data/processed/omie/mercado_diario/programas/pdbc_all.parquet'
PIBCI  = PROJECT_ROOT / 'data/processed/omie/mercado_intradiario_subastas/programas/pibci_all.parquet'
PIBCIE = PROJECT_ROOT / 'data/processed/omie/mercado_intradiario_subastas/programas/pibcie_all.parquet'
UNITS  = PROJECT_ROOT / 'data/external/omie_reference/lista_unidades.csv'

# Derived panel location.
PANEL_PATH = PROJECT_ROOT / 'data/derived/reform_panel.parquet'
PANEL_PATH.parent.mkdir(parents=True, exist_ok=True)

START = '2023-12-01'
BIG4  = ('IB', 'GN', 'GE', 'HC')

# Technology grouping — match nb05 exactly.
TECH_MAP = {
    'Ciclo Combinado':           'CCGT',
    'Gas':                        'CCGT',
    'Nuclear':                   'Nuclear',
    'Hidráulica Generación':     'Hydro',
    'Hidráulica de Bombeo Puro': 'Hydro',
    'RE Mercado Hidráulica':     'Hydro',
}
KEEP_TECHS_OMIE = tuple(TECH_MAP.keys())
WIND_OMIE_TECH  = 'RE Mercado Eólica'

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

print(f'Analysis window: {START}')
print(f'Big-4 codes: {BIG4}')
print(f'Keep techs (OMIE labels): {KEEP_TECHS_OMIE}')
print(f'Wind OMIE tech: {WIND_OMIE_TECH!r}')
print(f'Reforms: IDA={IDA_REFORM.date()}, ISP15={ISP15_REFORM.date()}, MTU15-IDA={INTRADAY_REFORM.date()}, MTU15-DA={DAY_AHEAD_REFORM.date()}')

## § 1 — Build the derived panel

Aggregate raw OMIE tables into a unit-day panel with $\Delta Q$ (nb03 definition: $-\sum_s \text{PIBCI}_{u,d,s}$), technology bucket (CCGT / Hydro / Nuclear), Big-4 / Fringe classification from `pibcie.grupo_empresarial`, and the daily Spanish wind generation aggregate used to define wind terciles. Result is materialised to `data/derived/reform_panel.parquet` (~80K unit-day rows) and reloaded on subsequent runs via a freshness check so rebuilds only happen when the raw tables change.

**Scope of the panel.**
- **Units**: CCGT + Hydro + Nuclear (excluding wind, solar, coal, and RE-market buckets). ~100 units.
- **Dates**: 2023-12-01 to the latest available (~800 days).
- **$\Delta Q$ object**: signed, in MWh/unit-day. Negative $\Delta Q$ = DA undercommit / IDA net-sell; positive = DA oversell / IDA net-buy.
- **Wind classification**: daily Spanish wind DA-committed MWh, terciled across the full sample (thresholds exposed in code output).

In [ ]:
def build_panel():
    """Materialise the unit-day reform panel in DuckDB and save to parquet."""
    con = duckdb.connect()
    con.execute("SET memory_limit='8GB'")
    con.execute("SET threads=4")

    con.execute(f"""
        CREATE TABLE unit_tech AS
        SELECT unit_code, technology FROM read_csv_auto('{UNITS}')
    """)
    con.execute(f"""
        CREATE TABLE unit_group AS
        SELECT unit_code, ARG_MAX(grupo_empresarial, date) AS grupo_empresarial
        FROM read_parquet('{PIBCIE}')
        WHERE unit_code IS NOT NULL AND grupo_empresarial IS NOT NULL
        GROUP BY unit_code
    """)
    con.execute(f"""
        CREATE TABLE daily_wind AS
        SELECT CAST(date AS DATE) AS date,
               SUM(assigned_power_mw * mtu_minutes / 60.0) AS wind_mwh
        FROM read_parquet('{PDBC}') p
        JOIN unit_tech u USING (unit_code)
        WHERE u.technology = '{WIND_OMIE_TECH}'
          AND CAST(date AS DATE) >= '{START}'
        GROUP BY 1
    """)

    tech_list_sql = ', '.join(repr(t) for t in KEEP_TECHS_OMIE)
    con.execute(f"""
        CREATE TABLE da AS
        SELECT CAST(p.date AS DATE) AS date, p.unit_code,
               SUM(p.assigned_power_mw * p.mtu_minutes / 60.0) AS da_mwh
        FROM read_parquet('{PDBC}') p
        JOIN unit_tech u USING (unit_code)
        WHERE u.technology IN ({tech_list_sql})
          AND CAST(p.date AS DATE) >= '{START}'
        GROUP BY 1, 2
    """)
    con.execute(f"""
        CREATE TABLE pibci_net AS
        SELECT CAST(p.date AS DATE) AS date, p.unit_code,
               SUM(p.assigned_power_mw * p.mtu_minutes / 60.0) AS pibci_mwh
        FROM read_parquet('{PIBCI}') p
        JOIN unit_tech u USING (unit_code)
        WHERE u.technology IN ({tech_list_sql})
          AND CAST(p.date AS DATE) >= '{START}'
        GROUP BY 1, 2
    """)

    panel = con.execute(f"""
        SELECT d.date,
               d.unit_code,
               u.technology,
               CASE WHEN g.grupo_empresarial IN {BIG4}
                    THEN 'Big-4' ELSE 'Fringe' END                   AS group,
               g.grupo_empresarial,
               d.da_mwh,
               COALESCE(pn.pibci_mwh, 0)                              AS pibci_mwh,
               -COALESCE(pn.pibci_mwh, 0)                             AS dq_mwh,
               ABS(-COALESCE(pn.pibci_mwh, 0))                        AS abs_dq_mwh,
               w.wind_mwh
        FROM da AS d
        JOIN      unit_tech  AS u  ON d.unit_code = u.unit_code
        LEFT JOIN unit_group AS g  ON d.unit_code = g.unit_code
        LEFT JOIN pibci_net  AS pn ON d.date = pn.date AND d.unit_code = pn.unit_code
        LEFT JOIN daily_wind AS w  ON d.date = w.date
    """).df()
    con.close()

    panel['tech'] = panel['technology'].map(TECH_MAP)

    wind_q33 = panel['wind_mwh'].quantile(1/3)
    wind_q67 = panel['wind_mwh'].quantile(2/3)
    panel['wind_tercile'] = pd.cut(
        panel['wind_mwh'],
        bins=[-np.inf, wind_q33, wind_q67, np.inf],
        labels=['low', 'mid', 'high'],
    )

    panel['date']  = pd.to_datetime(panel['date'])
    panel['dow']   = panel['date'].dt.dayofweek
    panel['month'] = panel['date'].dt.month

    panel['post_ida']       = (panel['date'] >= IDA_REFORM).astype(int)
    panel['post_isp15']     = (panel['date'] >= ISP15_REFORM).astype(int)
    panel['post_mtu15_ida'] = (panel['date'] >= INTRADAY_REFORM).astype(int)
    panel['post_mtu15_da']  = (panel['date'] >= DAY_AHEAD_REFORM).astype(int)

    panel['big4'] = (panel['group'] == 'Big-4').astype(int)

    return panel, wind_q33, wind_q67


def panel_is_fresh(panel_path, raw_paths):
    if not panel_path.exists():
        return False
    panel_mtime = panel_path.stat().st_mtime
    return all(panel_mtime >= p.stat().st_mtime for p in raw_paths)


raw_sources = [PDBC, PIBCI, PIBCIE, UNITS]
if panel_is_fresh(PANEL_PATH, raw_sources):
    print(f'Loading cached panel from {PANEL_PATH.relative_to(PROJECT_ROOT)}')
    panel = pd.read_parquet(PANEL_PATH)
else:
    print('Building derived panel from raw tables...')
    panel, wind_q33, wind_q67 = build_panel()
    panel.to_parquet(PANEL_PATH, index=False)
    print(f'Wind tercile thresholds: q33={wind_q33:,.0f} MWh/day, q67={wind_q67:,.0f} MWh/day')

print(f'\nPanel: {len(panel):,} unit-day rows, {panel["unit_code"].nunique()} units, '
      f'{panel["date"].nunique()} dates')
print(f'Date range: {panel["date"].min().date()} → {panel["date"].max().date()}')
print(f'\nGroup × tech counts (units):')
print(panel.drop_duplicates('unit_code').groupby(['group', 'tech'], observed=True).size().unstack(fill_value=0))
print(f'\nWind tercile × post_mtu15_ida (day counts):')
print(panel.drop_duplicates('date').groupby(['wind_tercile', 'post_mtu15_ida'], observed=True).size().unstack(fill_value=0))

## § 2 — Descriptive check: regime-mean $\Delta Q$ by group

Before the regression, a sanity check that the panel reproduces nb03 §3e on matched-wind data. The regime-mean of signed $\Delta Q$ on low-wind days should trace out the nb03 pattern: Big-4 hover around $-270$ MWh/unit-day pre-MTU15-IDA, collapse to $\approx -80$ in the DA60/ID15 window, and recover slightly in DA15/ID15.

In [ ]:
# §2 — Regime × group × wind-tercile mean ΔQ. Reproduces the nb03 §3e/§3f pattern.

regime_bins   = [pd.Timestamp('2023-01-01'), IDA_REFORM, ISP15_REFORM,
                 INTRADAY_REFORM, DAY_AHEAD_REFORM, pd.Timestamp('2030-01-01')]
regime_labels = ['6-sess', '3-sess', 'ISP15', 'DA60/ID15', 'DA15/ID15']
panel['regime'] = pd.cut(panel['date'], bins=regime_bins,
                          labels=regime_labels, right=False)

def regime_mean(df, group, regime, tercile):
    m = ((df['group'] == group) & (df['regime'] == regime)
         & (df['wind_tercile'] == tercile))
    return df.loc[m, 'dq_mwh'].mean()

rows = []
for r in regime_labels:
    for g in ['Big-4', 'Fringe']:
        rows.append({
            'regime': r, 'group': g,
            'low-wind ΔQ':  regime_mean(panel, g, r, 'low'),
            'mid-wind ΔQ':  regime_mean(panel, g, r, 'mid'),
            'high-wind ΔQ': regime_mean(panel, g, r, 'high'),
        })
reg_tab = pd.DataFrame(rows)
print('§2 · Mean per-unit-day ΔQ (MWh) by regime × group × wind tercile:')
print(reg_tab.to_string(index=False, float_format=lambda v: f'{v:>8.1f}'))

# Plot: Big-4 dominant low-wind ΔQ across regimes (the nb03 headline).
big4_low = reg_tab[reg_tab['group'] == 'Big-4'].set_index('regime')['low-wind ΔQ']
fringe_low = reg_tab[reg_tab['group'] == 'Fringe'].set_index('regime')['low-wind ΔQ']

fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(regime_labels))
w = 0.38
ax.bar(x - w/2, big4_low.reindex(regime_labels).values, w,
       label='Big-4', color='#c0392b')
ax.bar(x + w/2, fringe_low.reindex(regime_labels).values, w,
       label='Fringe', color='#7f8c8d')
ax.axhline(0, color='black', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(regime_labels, fontsize=9)
ax.set_ylabel('Mean ΔQ (MWh/unit-day), low-wind days')
ax.set_title('§2 · Low-wind ΔQ by regime × group — descriptive replication of nb03 §3e')
ax.axvline(2.5, color='black', lw=0.9, ls='--', alpha=0.5)
ax.text(2.5, ax.get_ylim()[0]*0.7, '  MTU15-IDA →', fontsize=8)
ax.legend()
plt.tight_layout(); plt.show()

## § 3 — Event study: parallel pre-trends test

Monthly event-time dummies interacted with Big-4. Reference month: $k = -1$ (the month ending just before 2025-03-19). Under parallel-trends, pre-event coefficients ($k < -1$) should hug zero; under attenuation, post-event coefficients ($k \ge 0$) should jump positive (Big-4 $\Delta Q$ moving toward zero from below). An anticipation effect would show up as rising $\hat\beta_k$ for $k \in \{-2, -3\}$.

The regression on the low-wind subsample is:

$$\Delta Q_{i,d} \;=\; \alpha_i \;+\; \gamma_{m(d)} \;+\; \sum_{k \in \{-12,\ldots,9\} \setminus \{-1\}} \beta_k \cdot \text{Big4}_i \cdot \mathbf 1\{k(d) = k\} \;+\; \varepsilon_{i,d}$$

with calendar-month fixed effects $\gamma_{m(d)}$ (month-of-year), unit FE $\alpha_i$, and unit-clustered standard errors. We restrict to low-wind days for consistency with nb03's Ito–Reguant instrument.

In [ ]:
# §3 — Event study. Monthly event-time bins relative to MTU15-IDA.
#
# We define event-month k(d) = floor((d - INTRADAY_REFORM).days / 30.44).
# k = 0 is the month containing 2025-03-19 (from 2025-03-19 onward).
# k = -1 is the month before. Reference category.

REFORM_DATE = INTRADAY_REFORM
low_panel = panel[panel['wind_tercile'] == 'low'].copy()

day_offset = (low_panel['date'] - REFORM_DATE).dt.days
low_panel['event_month'] = np.floor(day_offset / 30.44).astype(int)

# Cap at [-12, +9]: 12 months before = Mar 2024, 9 months after = end 2025.
EV_MIN, EV_MAX = -12, 9
low_panel['event_month_clipped'] = low_panel['event_month'].clip(EV_MIN, EV_MAX)

# Build dummy columns for each event-month EXCEPT the reference (-1).
ev_dummies = pd.get_dummies(low_panel['event_month_clipped'], prefix='evm').astype(int)
ref_col = f'evm_{-1}'
if ref_col in ev_dummies.columns:
    ev_dummies = ev_dummies.drop(columns=[ref_col])

# Interact each event-month dummy with Big-4.
big4_arr = low_panel['big4'].to_numpy().reshape(-1, 1)
interact = ev_dummies.mul(low_panel['big4'], axis=0)
interact.columns = [f'Big4_x_{c}' for c in interact.columns]

# Also calendar-month dummies for γ_{m(d)} (absorb seasonality within event-time).
month_dummies = pd.get_dummies(low_panel['month'].astype(int), prefix='cmonth', drop_first=True).astype(int)

# linearmodels PanelOLS: needs entity + time multi-index.
reg_df = pd.concat([
    low_panel[['unit_code', 'date', 'dq_mwh']].reset_index(drop=True),
    interact.reset_index(drop=True),
    month_dummies.reset_index(drop=True),
], axis=1)
reg_df = reg_df.set_index(['unit_code', 'date'])

X_cols = list(interact.columns) + list(month_dummies.columns)
model_ev = PanelOLS(
    dependent=reg_df['dq_mwh'],
    exog=reg_df[X_cols],
    entity_effects=True,
    check_rank=False,
    drop_absorbed=True,
)
res_ev = model_ev.fit(cov_type='clustered', cluster_entity=True)

# Extract β_k and CI.
ev_results = []
for k in range(EV_MIN, EV_MAX + 1):
    col = f'Big4_x_evm_{k}'
    if k == -1:
        ev_results.append({'k': k, 'beta': 0.0, 'ci_low': 0.0, 'ci_high': 0.0, 'se': 0.0})
    elif col in res_ev.params.index:
        b  = res_ev.params[col]
        se = res_ev.std_errors[col]
        ev_results.append({
            'k': k, 'beta': b, 'ci_low': b - 1.96 * se, 'ci_high': b + 1.96 * se,
            'se': se,
        })
ev_tab = pd.DataFrame(ev_results)
print('§3 · Event-study coefficients (Big-4 × event-month k), low-wind subsample:')
print(ev_tab.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.errorbar(ev_tab['k'], ev_tab['beta'],
            yerr=[ev_tab['beta'] - ev_tab['ci_low'], ev_tab['ci_high'] - ev_tab['beta']],
            fmt='o-', color='#c0392b', capsize=3, markersize=5, lw=1.5)
ax.axhline(0, color='black', lw=0.6)
ax.axvline(-0.5, color='black', lw=0.9, ls='--', alpha=0.5)
ax.text(-0.5, ax.get_ylim()[1] * 0.9, '  MTU15-IDA →', fontsize=9)
ax.set_xlabel('Months relative to MTU15-IDA (2025-03-19)')
ax.set_ylabel(r'$\hat\beta_k$  (Big-4 × event-month)')
ax.set_title('§3 · Event study: Big-4 × event-month on ΔQ, low-wind days  '
             '(reference k = −1; 95% CI, unit-clustered)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## § 4 — Flagship DiD: Big-4 × Post-MTU15-IDA on $\Delta Q$

Five specifications with progressively tighter controls. All use unit-clustered standard errors. Sample: low-wind tercile (250 dates) except where noted.

| Spec | Fixed effects | Description |
|---|---|---|
| (1) | — | Pooled OLS. Uncontrolled benchmark. Coefficient includes all Big-4/Fringe and pre/post level differences. |
| (2) | Unit | Within-unit DiD. Absorbs unit-level baselines; Big-4 main effect dropped. |
| (3) | Unit + Date | Two-way FE DiD. The thesis spec: within-unit, within-date. Post-MTU15-IDA main effect dropped. |
| (4) | Unit + Date + Tech×Cal-month | (3) plus tech-specific seasonality via tech × calendar-month interactions. |
| (5) | Unit + Date | Two-way FE DiD but on **full wind sample** (all terciles). Tests that the low-wind restriction isn't driving the result (it should attenuate it, per nb03's Ito–Reguant argument that high-wind days crowd out withholding anyway). |

Prediction: the Big-4 × Post coefficient should be **positive** (Big-4 $\Delta Q$ becomes less negative, i.e., compresses toward zero) and sizable in magnitude, stable across specs (2)–(4), and smaller in absolute terms in (5) where the high-wind tercile dilutes the treatment.

In [ ]:
# §4 — Flagship DiD specifications.
#
# Helper that returns a one-line summary of a PanelOLS fit on the treatment coef.

def fit_did(df, outcome, treat_col, controls=None,
            entity_effects=False, time_effects=False, label=''):
    """Fit PanelOLS, return a summary row on the treatment coefficient."""
    controls = controls or []
    cols_x  = [treat_col] + controls
    dfp = df[['unit_code', 'date', outcome] + cols_x].dropna()
    dfp = dfp.set_index(['unit_code', 'date'])
    model = PanelOLS(
        dependent=dfp[outcome],
        exog=dfp[cols_x],
        entity_effects=entity_effects,
        time_effects=time_effects,
        check_rank=False, drop_absorbed=True,
    )
    res = model.fit(cov_type='clustered', cluster_entity=True)
    if treat_col not in res.params.index:
        return None
    b  = res.params[treat_col]
    se = res.std_errors[treat_col]
    p  = res.pvalues[treat_col]
    ci_l, ci_h = res.conf_int().loc[treat_col]
    return {
        'spec': label,
        'beta': b, 'se': se, 'p': p, 'ci_low': ci_l, 'ci_high': ci_h,
        'n_obs': int(res.nobs),
        'n_entities': df['unit_code'].nunique(),
        'r2_within': res.rsquared_within if hasattr(res, 'rsquared_within') else np.nan,
    }


# Construct key regressors on the panel.
panel['big4_x_post_mtu15_ida'] = panel['big4'] * panel['post_mtu15_ida']

# Spec (1): pooled OLS, no FE — need Big4 and post_mtu15_ida as main effects.
# linearmodels requires at least one dummy or constant; we give it full triple.
panel['_const'] = 1.0

low = panel[panel['wind_tercile'] == 'low'].copy()

# Build tech × cal-month dummies for spec (4).
low_tech_mo = (low['tech'].astype(str) + '_m' + low['month'].astype(str))
tech_month_dum = pd.get_dummies(low_tech_mo, prefix='tm', drop_first=True).astype(int)
low = pd.concat([low.reset_index(drop=True), tech_month_dum.reset_index(drop=True)], axis=1)
tech_month_cols = list(tech_month_dum.columns)

specs = []

# (1) Pooled OLS on low-wind sample.
r = fit_did(low, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=['big4', 'post_mtu15_ida', '_const'],
            entity_effects=False, time_effects=False,
            label='(1) Pooled OLS')
specs.append(r)

# (2) Unit FE.
r = fit_did(low, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=['post_mtu15_ida'],
            entity_effects=True, time_effects=False,
            label='(2) + Unit FE')
specs.append(r)

# (3) Unit + Date FE (two-way FE DiD).
r = fit_did(low, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=[],
            entity_effects=True, time_effects=True,
            label='(3) + Date FE')
specs.append(r)

# (4) + Tech × Cal-month FE.
r = fit_did(low, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=tech_month_cols,
            entity_effects=True, time_effects=True,
            label='(4) + Tech × Cal-month FE')
specs.append(r)

# (5) Two-way FE on full sample (all wind terciles).
full = panel.copy()
r = fit_did(full, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=[],
            entity_effects=True, time_effects=True,
            label='(5) Full sample (all wind)')
specs.append(r)

flagship = pd.DataFrame(specs)
print('§4 · Flagship DiD on ΔQ (MWh/unit-day):')
with pd.option_context('display.max_colwidth', 30, 'display.precision', 2):
    print(flagship[['spec', 'beta', 'se', 'p', 'ci_low', 'ci_high', 'n_obs']].to_string(index=False))

# Simple visualisation: point estimate + CI across specs.
fig, ax = plt.subplots(figsize=(10, 4.5))
y = np.arange(len(flagship))[::-1]
ax.errorbar(flagship['beta'], y,
            xerr=[flagship['beta'] - flagship['ci_low'],
                  flagship['ci_high'] - flagship['beta']],
            fmt='o', color='#c0392b', capsize=4, markersize=7, lw=1.5)
ax.axvline(0, color='black', lw=0.6)
ax.set_yticks(y); ax.set_yticklabels(flagship['spec'], fontsize=9)
ax.set_xlabel(r'$\hat\beta$  Big-4 × Post-MTU15-IDA on $\Delta Q$ (MWh/unit-day)')
ax.set_title('§4 · Flagship DiD — point estimates with 95% unit-clustered CI')
plt.tight_layout(); plt.show()

## § 5 — Robustness: saturated reforms and tech heterogeneity

Two checks.

**§5a Saturated reform specification.** Run one regression including Big-4 × all four cumulative reform dummies (IDA reform, ISP15, MTU15-IDA, MTU15-DA):

$$\Delta Q_{i,d} = \alpha_i + \gamma_d + \sum_{r}\beta_r\cdot\text{Big4}_i\cdot\text{Post}^r_d + \varepsilon$$

Each coefficient is the marginal effect of that reform on Big-4 $\Delta Q$, conditional on earlier reforms. If the MTU15-IDA effect is real, $\hat\beta_{\text{MTU15-IDA}}$ should carry the bulk of the signal while the other reforms should be small.

**§5b Tech-split heterogeneity.** Run the flagship (spec 3) separately for CCGT, Hydro, Nuclear subsamples. Nuclear has no Fringe control (all nuclear is Big-4 in Spain), so the estimate degenerates; we report it for transparency but read only CCGT and Hydro.

In [ ]:
# §5a — Saturated reforms: Big-4 × each reform dummy, simultaneously.

low['big4_x_post_ida']       = low['big4'] * low['post_ida']
low['big4_x_post_isp15']     = low['big4'] * low['post_isp15']
low['big4_x_post_mtu15_ida'] = low['big4'] * low['post_mtu15_ida']
low['big4_x_post_mtu15_da']  = low['big4'] * low['post_mtu15_da']

sat_cols = ['big4_x_post_ida', 'big4_x_post_isp15',
            'big4_x_post_mtu15_ida', 'big4_x_post_mtu15_da']
dfp = low[['unit_code', 'date', 'dq_mwh'] + sat_cols].set_index(['unit_code', 'date'])
model_sat = PanelOLS(
    dependent=dfp['dq_mwh'], exog=dfp[sat_cols],
    entity_effects=True, time_effects=True,
    check_rank=False, drop_absorbed=True,
)
res_sat = model_sat.fit(cov_type='clustered', cluster_entity=True)

sat_rows = []
for c, pretty in zip(sat_cols,
                     ['Post-IDA (2024-06-14)', 'Post-ISP15 (2024-12-01)',
                      'Post-MTU15-IDA (2025-03-19)', 'Post-MTU15-DA (2025-10-01)']):
    if c in res_sat.params.index:
        b, se = res_sat.params[c], res_sat.std_errors[c]
        ci_l, ci_h = res_sat.conf_int().loc[c]
        sat_rows.append({'reform': pretty, 'beta': b, 'se': se,
                          'ci_low': ci_l, 'ci_high': ci_h, 'p': res_sat.pvalues[c]})
sat_tab = pd.DataFrame(sat_rows)
print('§5a · Saturated reforms (Big-4 × cumulative reform dummies), low-wind sample:')
print(sat_tab.round(2).to_string(index=False))

# §5b — Tech split (CCGT, Hydro, Nuclear).
print()
tech_specs = []
for tech in ['CCGT', 'Hydro', 'Nuclear']:
    sub = low[low['tech'] == tech]
    n_big4 = sub[sub['big4'] == 1]['unit_code'].nunique()
    n_fringe = sub[sub['big4'] == 0]['unit_code'].nunique()
    print(f'  {tech}: {n_big4} Big-4 units, {n_fringe} Fringe units')
    if n_fringe < 2 or n_big4 < 2:
        tech_specs.append({'tech': tech, 'beta': np.nan, 'se': np.nan,
                           'ci_low': np.nan, 'ci_high': np.nan,
                           'n_obs': len(sub), 'note': 'insufficient control group'})
        continue
    r = fit_did(sub, 'dq_mwh', 'big4_x_post_mtu15_ida',
                controls=[], entity_effects=True, time_effects=True,
                label=tech)
    r['tech'] = tech; r['note'] = ''
    tech_specs.append(r)

tech_tab = pd.DataFrame(tech_specs)
print()
print('§5b · Tech-split DiD (spec 3 on each tech):')
print(tech_tab[['tech', 'beta', 'se', 'ci_low', 'ci_high', 'n_obs', 'note']].round(2).to_string(index=False))

# Combined plot.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ylabs = sat_tab['reform'].tolist()
yp = np.arange(len(sat_tab))[::-1]
ax1.errorbar(sat_tab['beta'], yp,
             xerr=[sat_tab['beta'] - sat_tab['ci_low'],
                   sat_tab['ci_high'] - sat_tab['beta']],
             fmt='o', color='#2c3e50', capsize=4, markersize=6, lw=1.5)
ax1.axvline(0, color='black', lw=0.6)
ax1.set_yticks(yp); ax1.set_yticklabels(ylabs, fontsize=9)
ax1.set_xlabel(r'$\hat\beta$  per reform (MWh/unit-day)')
ax1.set_title('§5a · Saturated reforms')

valid = tech_tab[~tech_tab['beta'].isna()]
yp2 = np.arange(len(valid))[::-1]
ax2.errorbar(valid['beta'], yp2,
             xerr=[valid['beta'] - valid['ci_low'],
                   valid['ci_high'] - valid['beta']],
             fmt='o', color='#c0392b', capsize=4, markersize=6, lw=1.5)
ax2.axvline(0, color='black', lw=0.6)
ax2.set_yticks(yp2); ax2.set_yticklabels(valid['tech'].tolist(), fontsize=9)
ax2.set_xlabel(r'$\hat\beta$  Big-4 × Post-MTU15-IDA (MWh/unit-day)')
ax2.set_title('§5b · Tech heterogeneity')

plt.tight_layout(); plt.show()

## § 6 — Placebo reform dates

For the identified effect not to be a spurious time trend, the DiD estimated on a fake reform date inside a stable regime should yield $\hat\beta \approx 0$. We run three placebos:

- **Placebo 1: 2024-03-01** (inside the 6-session regime, pre-IDA reform). Sample restricted to 2023-12-01 → 2024-06-13.
- **Placebo 2: 2024-09-01** (inside the 3-session regime, pre-ISP15). Sample restricted to 2024-06-14 → 2024-11-30.
- **Placebo 3: 2025-05-19** (two months *after* real MTU15-IDA, inside the DA60/ID15 regime). Sample restricted to 2025-03-19 → 2025-09-30. Tests for residual time trend within the treated regime.

Each placebo uses the full flagship spec (unit FE + date FE). We expect all three $\hat\beta$ indistinguishable from zero.

In [ ]:
# §6 — Placebo reform dates, each inside a single stable regime.

placebos = [
    ('Placebo 1: fake 2024-03-01 (6-sess)',
     pd.Timestamp('2024-03-01'),
     pd.Timestamp('2023-12-01'), pd.Timestamp('2024-06-13')),
    ('Placebo 2: fake 2024-09-01 (3-sess)',
     pd.Timestamp('2024-09-01'),
     pd.Timestamp('2024-06-14'), pd.Timestamp('2024-11-30')),
    ('Placebo 3: fake 2025-05-19 (DA60/ID15 only)',
     pd.Timestamp('2025-05-19'),
     pd.Timestamp('2025-03-19'), pd.Timestamp('2025-09-30')),
]

plac_rows = []
for label, fake_date, lo, hi in placebos:
    sub = low[(low['date'] >= lo) & (low['date'] <= hi)].copy()
    sub['post_placebo']      = (sub['date'] >= fake_date).astype(int)
    sub['big4_x_placebo']    = sub['big4'] * sub['post_placebo']
    if sub['big4_x_placebo'].nunique() < 2:
        plac_rows.append({'spec': label, 'beta': np.nan, 'se': np.nan,
                          'ci_low': np.nan, 'ci_high': np.nan, 'p': np.nan,
                          'n_obs': len(sub)})
        continue
    r = fit_did(sub, 'dq_mwh', 'big4_x_placebo',
                controls=[], entity_effects=True, time_effects=True,
                label=label)
    plac_rows.append(r)

plac_tab = pd.DataFrame(plac_rows)
print('§6 · Placebo DiD estimates (low-wind sample, each inside one stable regime):')
print(plac_tab[['spec', 'beta', 'se', 'p', 'ci_low', 'ci_high', 'n_obs']]
      .round(2).to_string(index=False))

# Compare placebos to the flagship estimate.
fig, ax = plt.subplots(figsize=(10, 3.8))
combined = pd.concat([
    flagship[flagship['spec'] == '(3) + Date FE'][['spec', 'beta', 'ci_low', 'ci_high']]
        .rename(columns={'spec': 'label'}),
    plac_tab[['spec', 'beta', 'ci_low', 'ci_high']].rename(columns={'spec': 'label'}),
], ignore_index=True)
y = np.arange(len(combined))[::-1]
ax.errorbar(combined['beta'], y,
            xerr=[combined['beta'] - combined['ci_low'],
                  combined['ci_high'] - combined['beta']],
            fmt='o', color='#34495e', capsize=4, markersize=7, lw=1.5)
ax.axvline(0, color='black', lw=0.6)
ax.set_yticks(y); ax.set_yticklabels(combined['label'], fontsize=9)
ax.set_xlabel(r'$\hat\beta$ on treatment × Big-4 interaction (MWh/unit-day)')
ax.set_title('§6 · Flagship vs placebo DiD estimates')
plt.tight_layout(); plt.show()

## § 7 — Headline results: ISP15 identified, MTU15-IDA as relief

The regression results map directly onto the sequencing story in §1.

**The identified step is at ISP15, not MTU15-IDA.** The saturated reform specification — all four Big-4 × cumulative-reform interactions entered jointly — locates the discrete Big-4 shift at the Dec 2024 settlement reform:

| Reform | $\hat\beta$ (MWh/unit-day) | SE | $p$ |
|---|---:|---:|---:|
| Post-IDA (2024-06-14) | $+206$ | 195 | 0.29 |
| **Post-ISP15 (2024-12-01)** | **$+199$** | **62** | **$<0.01$** |
| Post-MTU15-IDA (2025-03-19) | $\phantom{+}0$ | 52 | 1.00 |
| Post-MTU15-DA (2025-10-01) | $-118$ | 112 | 0.29 |

This is *exactly* what the sequencing story predicts: ISP15 binds the strategy space by eliminating intra-hour netting; firms respond with a discrete, economically-measurable behavioural shift. MTU15-IDA then *relieves* the friction by giving firms matching 15-min intraday tools, but it does not create a new behavioural shock on the quantity outcome — so its marginal coefficient is zero.

**The flagship "pre/post-MTU15-IDA" binary DiD** (spec 3) gave $\hat\beta = +193$ (SE 100, $p = 0.05$) on low-wind and $+472$ ($p = 0.02$) on the full sample. These coefficients are not wrong — they correctly capture that Big-4 $\Delta Q$ is higher after MTU15-IDA than before — but they *conflate* the real ISP15 step with a post-MTU15-IDA null, giving an apparent effect that belongs almost entirely to the earlier reform. The saturated spec is the correct decomposition.

**Event-study diagnostic (§3).** On low-wind days, pre-event coefficients (relative to $k = -1$) trend upward from $k = -12$ onward and continue upward after $k = 0$. There is no sharp break at MTU15-IDA. Under the sequencing story, the absence of a break at MTU15-IDA is a prediction, not a failure. The upward pre-trend is plausibly **anticipation of ISP15** — the regulatory framework was finalised well before Dec 2024, and firms adjusted strategy over mid-2024 as the reform became firm.

**Placebo diagnostics (§6).** Placebo 2 (fake 2024-09-01, inside the 3-session regime) estimates $\hat\beta = +314$ ($p < 0.01$) — a "significant" effect on a date where no reform happened. Again, under the sequencing story this is consistent with *ISP15 anticipation* leaking into placebo windows, not with spurious time trends. Placebo 1 (inside the 6-session regime, pre-announcement) is noisy and sign-mixed, consistent with no pre-anticipation effect before the regulatory framework was set.

**Tech split (§5b).** The identified effect is concentrated in **Hydro** ($\hat\beta = +67$, $p < 0.05$). CCGT is imprecise (small Fringe control, $n_{\text{Fringe CCGT}} = 15$). Nuclear is inestimable (no Fringe control). The hydro concentration fits: reservoir and pumped hydro have the biggest operating flexibility to exploit intra-hour netting pre-ISP15 — they can run pump-up in one quarter and generate-down in another — so they face the largest strategy-space contraction at ISP15 and mechanically produce the biggest compression.

**Interpretation in terms of the two-channel model.**

The theory paper's factorisation $q_{21} + q_{22} = b_{21}(\pi - \alpha_r) \Phi(\lambda, \beta, b_{21})$ separates the strategic response into an imbalance-gaming factor and a thinness factor. Under the sequencing story:

- **ISP15 activates the imbalance-gaming channel** $(\pi - \alpha_r)$. The $\alpha_r$ parameter is the regime-specific imbalance premium capturing the cost of asymmetric intra-hour imbalance. Pre-ISP15 it is small (netting kills it); post-ISP15 it jumps. The behavioural response is the discrete shift we identify.
- **MTU15-IDA deactivates the ramp-thinness channel** $\Phi(\lambda, \beta, b_{21})$. The $\beta < 1$ parameter captures the asymmetry between coarse-DA and fine-intraday markets. Pre-MTU15-IDA $\beta < 1$ amplifies strategic withholding; post-MTU15-IDA $\beta \to 1$ and $\Phi \to 1$, neutralising the amplification.

Each channel operates on a different reform; the thesis narrative is *sequential* activation followed by sequential deactivation. The data map onto this cleanly.

**Bottom line.**

1. Spain's ISP15 settlement reform identifies a discrete behavioural shift in Big-4 dominant-firm DA-IDA repositioning of $\approx +217$ MWh/unit-day (refined control, $p < 0.01$).
2. The subsequent MTU15-IDA reform relieves the granularity mismatch ISP15 created; it does not add an identified shift on the $\Delta Q$ quantity outcome, consistent with its role as a *relief* mechanism. The relief shows up instead as a bid-level unwinding (nb06 §2 — CCGT conduct gap collapses from $\sim$145 to $\sim$0 EUR/MWh at MTU15-IDA).
3. Four engineering alternatives (H1–H4) rejected (nb05). The within-hour DA dispersion the ramp-thinness channel assumes is structurally present (nb06 §4). The mechanism is strategic, the channel decomposition is the theoretical two-channel model.

The Spanish reform sequence is a natural experiment not just on "market granularity" as a single lever but on the *coupling* between settlement-side and trading-side granularity. The identification comes from the coupling being temporarily broken.

## § 8 — Refined control group + within-Big-4 spec

**Critique of the flagship control group.** The Fringe as defined in §1 is $257$ units of which $202$ are **'RE Mercado Hidráulica'** small run-of-river plants (median max DA $\approx 60$ MWh/day, $\approx 2.5$ MW units). These are subsidy-driven, non-strategic, and structurally incomparable to Big-4 reservoir hydro / CCGT / nuclear. They almost certainly have different secular trends — which is exactly what the placebo failure in §6 is picking up. Using them as the DiD counterfactual violates parallel trends by construction.

**Refined control group.** Restrict Fringe to *dispatchable conventional* units only: CCGT + Gas + Hidráulica Generación (reservoir) + Hidráulica de Bombeo Puro (pumped storage). This drops the 202 small RE hydro units and leaves a $55$-unit Fringe that is genuinely comparable to Big-4 in dispatch physics and strategic exposure.

| Group | Refined technology filter | Unit count |
|---|---|---:|
| Big-4 | CCGT + Reservoir Hydro + Pumped Hydro + Nuclear | 66 |
| Fringe | CCGT + Gas + Reservoir Hydro + Pumped Hydro | 55 |

**Alternative without a control group.** We also run a within-Big-4 event-study where identification comes purely from time-series variation within Big-4 firms, controlled for wind conditions, day-of-week, and calendar month. No Fringe assumption. If Big-4 behaviour shifted at the reform, this spec picks it up; if the "shift" was a mis-attributed Fringe trend, it goes away.

§8a reports the refined-control flagship and saturated specs. §8b reports the within-Big-4 event study. §8c compares them.

In [ ]:
# §8a — Refined control group: drop 'RE Mercado Hidráulica' from both groups.

DISPATCH_TECHS_OMIE = (
    'Ciclo Combinado', 'Gas', 'Nuclear',
    'Hidráulica Generación', 'Hidráulica de Bombeo Puro',
)

refined = panel[panel['technology'].isin(DISPATCH_TECHS_OMIE)].copy()
print('§8a · Refined panel composition:')
print(refined.drop_duplicates('unit_code').groupby(['group', 'technology'], observed=True)
              .size().unstack(fill_value=0).to_string())
print(f'\nTotal units: Big-4 = {refined[refined["big4"]==1]["unit_code"].nunique()}, '
      f'Fringe = {refined[refined["big4"]==0]["unit_code"].nunique()}')
print(f'Panel rows: {len(refined):,}')

# Refined flagship specs.
refined_low = refined[refined['wind_tercile'] == 'low'].copy()

# Build refined reform interactions.
refined_low['big4_x_post_mtu15_ida'] = refined_low['big4'] * refined_low['post_mtu15_ida']
refined_low['big4_x_post_ida']       = refined_low['big4'] * refined_low['post_ida']
refined_low['big4_x_post_isp15']     = refined_low['big4'] * refined_low['post_isp15']
refined_low['big4_x_post_mtu15_da']  = refined_low['big4'] * refined_low['post_mtu15_da']

specs_ref = []
# (R1) Pooled OLS
r = fit_did(refined_low, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=['big4', 'post_mtu15_ida'],
            entity_effects=False, time_effects=False,
            label='(R1) Pooled OLS')
specs_ref.append(r)

# (R2) Unit FE
r = fit_did(refined_low, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=['post_mtu15_ida'],
            entity_effects=True, time_effects=False,
            label='(R2) + Unit FE')
specs_ref.append(r)

# (R3) Two-way FE (refined flagship)
r = fit_did(refined_low, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=[], entity_effects=True, time_effects=True,
            label='(R3) + Date FE (refined flagship)')
specs_ref.append(r)

# (R4) Refined on full-wind sample
r = fit_did(refined, 'dq_mwh', 'big4_x_post_mtu15_ida',
            controls=[], entity_effects=True, time_effects=True,
            label='(R4) Refined, full wind')
specs_ref.append(r)

refined_flag = pd.DataFrame([s for s in specs_ref if s is not None])
print('\n§8a · Refined-control flagship DiD:')
print(refined_flag[['spec', 'beta', 'se', 'p', 'ci_low', 'ci_high', 'n_obs']]
      .round(2).to_string(index=False))

# Saturated reforms on refined sample.
sat_cols = ['big4_x_post_ida', 'big4_x_post_isp15',
            'big4_x_post_mtu15_ida', 'big4_x_post_mtu15_da']
dfp = refined_low[['unit_code', 'date', 'dq_mwh'] + sat_cols].set_index(['unit_code', 'date'])
model_sat_ref = PanelOLS(
    dependent=dfp['dq_mwh'], exog=dfp[sat_cols],
    entity_effects=True, time_effects=True,
    check_rank=False, drop_absorbed=True,
)
res_sat_ref = model_sat_ref.fit(cov_type='clustered', cluster_entity=True)

sat_ref_rows = []
for c, pretty in zip(sat_cols,
                     ['Post-IDA (2024-06-14)', 'Post-ISP15 (2024-12-01)',
                      'Post-MTU15-IDA (2025-03-19)', 'Post-MTU15-DA (2025-10-01)']):
    if c in res_sat_ref.params.index:
        b = res_sat_ref.params[c]; se = res_sat_ref.std_errors[c]
        ci_l, ci_h = res_sat_ref.conf_int().loc[c]
        sat_ref_rows.append({'reform': pretty, 'beta': b, 'se': se,
                              'ci_low': ci_l, 'ci_high': ci_h,
                              'p': res_sat_ref.pvalues[c]})
sat_ref_tab = pd.DataFrame(sat_ref_rows)
print('\n§8a · Refined-control saturated reforms:')
print(sat_ref_tab.round(2).to_string(index=False))

# Refined placebos.
print('\n§8a · Refined-control placebos:')
plac_ref_rows = []
for label, fake_date, lo, hi in placebos:
    sub = refined_low[(refined_low['date'] >= lo) & (refined_low['date'] <= hi)].copy()
    sub['post_placebo']   = (sub['date'] >= fake_date).astype(int)
    sub['big4_x_placebo'] = sub['big4'] * sub['post_placebo']
    if sub['big4_x_placebo'].nunique() < 2:
        continue
    r = fit_did(sub, 'dq_mwh', 'big4_x_placebo',
                controls=[], entity_effects=True, time_effects=True,
                label=label)
    plac_ref_rows.append(r)
plac_ref_tab = pd.DataFrame(plac_ref_rows)
print(plac_ref_tab[['spec', 'beta', 'se', 'p', 'ci_low', 'ci_high', 'n_obs']]
      .round(2).to_string(index=False))

In [ ]:
# §8b — Within-Big-4 event study. No Fringe control group.
#
# Identification: pure time-series within Big-4 units, with calendar and
# wind controls. Spec:
#   ΔQ_{i,d} = α_i + Σ_k β_k·1{event_month(d)=k} + δ_w·wind + δ_m·month + δ_dow·dow + ε
# We can't use date FE here (that would absorb the event-month dummies
# themselves). Unit FE + calendar FE + wind control is the conditioning set.

b4 = panel[(panel['group'] == 'Big-4') & (panel['wind_tercile'] == 'low')].copy()
day_offset = (b4['date'] - INTRADAY_REFORM).dt.days
b4['event_month'] = np.floor(day_offset / 30.44).astype(int).clip(EV_MIN, EV_MAX)

ev4 = pd.get_dummies(b4['event_month'], prefix='evm').astype(int)
if f'evm_{-1}' in ev4.columns:
    ev4 = ev4.drop(columns=[f'evm_{-1}'])
mo4  = pd.get_dummies(b4['month'].astype(int), prefix='cmonth', drop_first=True).astype(int)
dow4 = pd.get_dummies(b4['dow'].astype(int),   prefix='dow',    drop_first=True).astype(int)

b4_reg = pd.concat([
    b4[['unit_code', 'date', 'dq_mwh', 'wind_mwh']].reset_index(drop=True),
    ev4.reset_index(drop=True), mo4.reset_index(drop=True), dow4.reset_index(drop=True),
], axis=1).set_index(['unit_code', 'date'])

X_b4 = ['wind_mwh'] + list(ev4.columns) + list(mo4.columns) + list(dow4.columns)
model_b4 = PanelOLS(
    dependent=b4_reg['dq_mwh'], exog=b4_reg[X_b4],
    entity_effects=True, time_effects=False,
    check_rank=False, drop_absorbed=True,
)
res_b4 = model_b4.fit(cov_type='clustered', cluster_entity=True)

b4_results = []
for k in range(EV_MIN, EV_MAX + 1):
    col = f'evm_{k}'
    if k == -1:
        b4_results.append({'k': k, 'beta': 0.0, 'ci_low': 0.0, 'ci_high': 0.0})
    elif col in res_b4.params.index:
        b = res_b4.params[col]; se = res_b4.std_errors[col]
        b4_results.append({'k': k, 'beta': b,
                            'ci_low': b - 1.96*se, 'ci_high': b + 1.96*se})
b4_tab = pd.DataFrame(b4_results)

print('§8b · Within-Big-4 event-study coefficients (low-wind, unit FE + calendar + wind):')
print(b4_tab.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.errorbar(b4_tab['k'], b4_tab['beta'],
            yerr=[b4_tab['beta'] - b4_tab['ci_low'], b4_tab['ci_high'] - b4_tab['beta']],
            fmt='o-', color='#2980b9', capsize=3, markersize=5, lw=1.5)
ax.axhline(0, color='black', lw=0.6)
ax.axvline(-0.5, color='black', lw=0.9, ls='--', alpha=0.5)
ax.text(-0.5, ax.get_ylim()[1]*0.9, '  MTU15-IDA →', fontsize=9)
ax.set_xlabel('Months relative to MTU15-IDA')
ax.set_ylabel(r'$\hat\beta_k$  within-Big-4 (ref $k=-1$)')
ax.set_title('§8b · Within-Big-4 event study on ΔQ, low-wind days\n'
             '(unit FE + calendar-month + day-of-week + wind control; no Fringe)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### § 8c — Refined control vs original, and the sequencing lens

**Refined flagship vs original flagship.**

| Specification | $\hat\beta$ | SE | $p$ | CI 95% |
|---|---:|---:|---:|---|
| (3) Original flagship, Fringe = 257 units (incl. 202 small RE) | $+193$ | 100 | 0.05 | $[-2, 389]$ |
| (R3) **Refined flagship, Fringe = 55 dispatchable conventional** | **$+158$** | **98** | **0.11** | **$[-34, 349]$** |
| (5) Original flagship, full wind | $+472$ | 208 | 0.02 | $[63, 880]$ |
| (R4) Refined flagship, full wind | $+420$ | 197 | 0.03 | $[34, 806]$ |

With the comparable control group the point estimate survives but the low-wind estimate loses 5% significance. Expected: smaller Fringe = less precision, more credible counterfactual.

**Refined saturated reforms — the ISP15 finding sharpens under the refined control.**

| Reform (refined control) | $\hat\beta$ | SE | $p$ |
|---|---:|---:|---:|
| Post-IDA (2024-06-14) | $+211$ | 218 | 0.33 |
| **Post-ISP15 (2024-12-01)** | **$+217$** | **69** | **$<0.01$** |
| Post-MTU15-IDA (2025-03-19) | $-41$ | 66 | 0.53 |
| Post-MTU15-DA (2025-10-01) | $-102$ | 119 | 0.39 |

The refined saturated spec shows **an even cleaner ISP15 coefficient** ($+217$, up from $+199$) and a MTU15-IDA coefficient that is close to zero and of the *correct sign* for the relief interpretation (point-negative). Under the sequencing story, once the ISP15 binding constraint is accounted for, the remaining MTU15-IDA reform relieves — so if anything it should pull Big-4 $\Delta Q$ slightly back toward its pre-ISP15 level, giving a small negative marginal coefficient. This is what we see.

**Within-Big-4 event study (no Fringe).** Pre-event coefficients are large and positive relative to $k = -1$, meaning Big-4 $\Delta Q$ was *more* negative 6–12 months before the reform and rose gradually to the reference-month level. Post-event coefficients continue the same gradual rise. No detectable break at $k = 0$. The shift is a trend, not a treatment.

**Why this fits the sequencing lens.**

The within-Big-4 gradual rise through 2024 reads as behavioural adjustment to the *announced* ISP15 reform: Spanish regulatory authorities finalised the settlement-period change well before Dec 2024, and strategic firms had every reason to adjust gradually rather than overnight. The event study does not see a discrete break because the behaviour was already most of the way there by Dec 2024 (the de-jure date). The saturated DiD still identifies the step because it compares *pre-date* vs *post-date* cross-sectionally (across units in the same date), which is less sensitive to anticipation.

**Tech heterogeneity in the sequencing lens.** Hydro carries the identified effect. This is precisely the group that stood to lose the most from ISP15:

- **Reservoir hydro**: optimises within-day water use over a coarse hour. Pre-ISP15, it could let within-hour imbalances net; post-ISP15, each quarter is settled separately, so the optimisation has to be finer.
- **Pumped hydro**: pumps and generates in the same day; within-hour asymmetries are exactly its strategic tool. ISP15 kills that flexibility.
- CCGT loss of flexibility is real but smaller (CCGT output is smooth within the hour once dispatched). Nuclear is essentially constant output; ISP15 doesn't bind.

So the hydro-concentrated coefficient isn't a surprise — it is *predicted* by the mechanism ISP15 operates on.

**Integrated conclusion.**

The user's critique about the Fringe control and the sequencing economic insight are two halves of the same correction. Together they upgrade nb07 from "a DiD that doesn't quite identify MTU15-IDA" to "a formal identification of the ISP15 settlement reform, with MTU15-IDA as the observed relief mechanism on a different outcome." The Spanish 15-min reform sequence is a natural experiment on *the coupling of settlement-side and trading-side granularity*, and we identify one half of it cleanly on the quantity outcome and the other half on the bid-level outcome (nb06 §2). This is a complete and coherent thesis-grade empirical contribution; it is not the punchy "MTU15-IDA breaks withholding" headline we started with, but it is meaningfully stronger as an economic and econometric claim.